# 03/09 — Cell-type deconvolution of the BRICHOS Visium data (cell2location)

**Goal.** Estimate per-spot cell-type composition of the mouse Visium
sections, with the *immune / microglia* compartment resolved using the
Chhatbar et al. 2026 (*Nat Immunol*) **mouse CNS-immune taxonomy** as the
reference (Zenodo `10.5281/zenodo.16938034`; mouse GEO GSE304010).

**Why a combined reference.** The Chhatbar atlas is **CNS-immune-only**
(Microglia, macrophages/MDM/CAM, and lymphoid — T/NK/plasma;
`Subclass_label`, 12 identities). It has **no** neurons/astro/oligo/
vascular, so it cannot deconvolve a whole 55 µm Visium spot on its own
(the model would be forced to explain neuronal signal with immune states →
garbage proportions). So we build a **combined reference**:

> Linnarsson/Zeisel adolescent mouse CNS — `Class`, 7 major types
> **+** Chhatbar CNS-immune identities (`MYE:` prefix)
> = one reference with a unified `cell_type` column

The brain reference's own `Immune` class is **stripped** before grafting, so
the immune compartment is described only by the fine atlas identities.

**Compute / environment.** Runs on the peer (`KI-CLJJYKHFX7`, 16-core /
48 GB Apple Silicon, drive local) in the `c2l` conda env. **No CUDA GPU** →
cell2location trains on CPU; epoch counts are reduced and flagged.

**Honest caveats.**
1. *Cross-dataset batch.* Two references (Linnarsson + Chhatbar) are merged;
   we model `ref_source` as batch, but residual batch can bias signatures.
   Sanity-check that `MYE:Microglia` (Tmem119/Siglech/Olfml3) lands broadly
   across grey matter where microglia should.
2. *Coarse background.* The brain reference resolves only 6 non-immune
   major types (neurons not sub-typed); adequate to absorb non-immune
   signal, but don't over-read the non-immune proportions.
3. *Minority cell type.* Microglia are a small fraction of each spot;
   identity-level proportions are noisy. Lean on the aggregate `MYE:*`
   (immune) fraction and treat finer calls as hypothesis-generating.
4. *Identities vs states.* `Subclass_label` gives cell *identities*. The
   microglial functional *states* (`Module_label`: Surveillance /
   Inflammation / Phagocytosis / MHC / IFN / Proliferation /
   Neuroprotection / Cytokine — the BRICHOS-relevant axis) are microglia
   sub-states; probe those with signature scoring (03/08) or a second pass.

Outputs: combined reference h5ad, per-spot proportions in `.obs`
(`prop_<cell_type>`) and `.obsm`, spatial composition maps, and a
region×treatment composition table with BRI-vs-PBS shifts.

In [ ]:
from __future__ import annotations
import sys, os, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# --- Visium data (drive is local on this peer) -------------------------
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'

# --- reference inputs --------------------------------------------------
MIC_DIR    = ROOT / 'data' / 'external' / 'microglia_taxonomy'
# pre-subsampled immune atlas (13k cells, 12 Subclass identities), shipped
# from the laptop; falls back to exporting the RDS bundle if absent.
ATLAS_SUB  = MIC_DIR / 'atlas_immune_subsampled.h5ad'
ATLAS_RDS  = MIC_DIR / 'Mouse_Myeloid_atlas_v1.RDS'
ATLAS_BUNDLE = MIC_DIR / 'atlas_bundle'
# brain background reference: Linnarsson/Zeisel adolescent mouse CNS (local
# on the drive). 'Class' = 7 major types; we strip its Immune class and
# graft the fine atlas immune identities instead.
LINN_H5AD  = BASEDIR / 'linnarsson_adolescence_full.h5ad'

# --- derived / built artifacts -----------------------------------------
REF_DIR    = ROOT / 'data' / 'external' / 'deconv_reference'
REF_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_H5AD = REF_DIR / 'combined_reference.h5ad'
C2L_REF_DIR   = REF_DIR / 'c2l_regression'
C2L_VIS_DIR   = REF_DIR / 'c2l_spatial'

SAMPLE_KEY    = 'sample_id'
REGION_KEY    = 're_annotation_regions'
TREATMENT_KEY = 'treatment'

# --- results -----------------------------------------------------------
TBL = ROOT / 'results' / 'tables' / 'deconvolution'; TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'; FIG.mkdir(parents=True, exist_ok=True)

print('visium h5ad :', H5AD, '(exists:', H5AD.exists(), ')')
print('atlas (sub) :', ATLAS_SUB, '(exists:', ATLAS_SUB.exists(), ')')
print('linnarsson  :', LINN_H5AD, '(exists:', LINN_H5AD.exists(), ')')

## 0 · Dependencies (one-time)

cell2location pulls in `scvi-tools` + `torch`. This is a large install; run
once. On Apple Silicon / no-GPU this installs the CPU build of torch. For a
GPU run do this on a CUDA box instead. `cellxgene-census` is used to fetch
the Allen reference.

In [ ]:
INSTALL = False   # flip to True the first time
if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'cell2location', 'cellxgene-census'], check=True)

# report what is available (imports guarded so the ref-wrangling cells can
# run even before cell2location is installed)
for p in ['torch', 'scvi', 'cell2location', 'cellxgene_census']:
    try:
        import importlib.metadata as _m
        print(f'{p:18}', _m.version(p))
    except Exception:
        print(f'{p:18} MISSING (set INSTALL=True)')
try:
    import torch
    ACCELERATOR = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('accelerator:', ACCELERATOR)
except Exception:
    ACCELERATOR = 'cpu'

## 1 · Myeloid atlas: Seurat .RDS → AnnData

We export the Seurat object to a Matrix-Market bundle via
`utils/export_seurat_atlas.R` (no SeuratDisk needed — robust across Seurat
4/5), then load it. The R step prints the candidate annotation columns; we
pick the one holding the myeloid *state* labels.

In [ ]:
from utils import deconvolution as dv

# Preferred path: the pre-subsampled immune atlas shipped from the laptop
# (13k cells, raw counts in .X). Fallback: export the full Seurat RDS to an
# mtx bundle (needs R + the 1GB RDS present) and load that.
if ATLAS_SUB.exists():
    atlas = sc.read_h5ad(ATLAS_SUB)
    atlas.layers['counts'] = atlas.X.copy()
    print('loaded pre-subsampled atlas:', atlas.shape)
else:
    if not (ATLAS_BUNDLE / 'matrix.mtx.gz').exists():
        assert ATLAS_RDS.exists(), 'need atlas_immune_subsampled.h5ad or the RDS'
        print('exporting Seurat -> mtx bundle...')
        res = subprocess.run(['Rscript', str(ROOT / 'utils' / 'export_seurat_atlas.R'),
                              str(ATLAS_RDS), str(ATLAS_BUNDLE)],
                             capture_output=True, text=True)
        print(res.stdout[-2000:]); print(res.stderr[-2000:]); res.check_returncode()
    atlas = dv.load_mtx_bundle(ATLAS_BUNDLE)
    print('loaded atlas bundle:', atlas.shape)

print('obs columns:', list(atlas.obs.columns))

In [ ]:
# Confirmed annotation hierarchy in the atlas RDS metadata:
#   Subclass_label      12  cell IDENTITIES: Microglia (152k), MDM, CAM,
#                           CD8 TRM, cDC1, CD4 Th17, Granulocyte, TEx,
#                           MAIT, Plasmablasts, Treg, NK     <- USE THIS
#   Module_label         8  microglial functional states: Surveillance,
#                           Inflammation, Phagocytosis, MHC, IFN,
#                           Proliferation, Neuroprotection, Cytokine
#   Module_id_label     32  finer module split
#   relabelled_clusters 99  finest clusters (too fine for spot deconv)
#
# Subclass_label is the disjoint cell-IDENTITY axis -> best for proportion
# deconvolution. The functional MODULES are microglia sub-states; probe
# those with signature scoring (as in 03/08) or a second deconvolution pass
# restricted to each spot's myeloid fraction.
for c in ['Subclass_label', 'Module_label', 'Module_id_label', 'relabelled_clusters']:
    if c in atlas.obs:
        print(f'  {c:20} {atlas.obs[c].nunique():>4} levels')

MYELOID_LABEL_COL = 'Subclass_label'
print('\nusing MYELOID_LABEL_COL =', MYELOID_LABEL_COL)
print(atlas.obs[MYELOID_LABEL_COL].value_counts())

## 2 · Linnarsson/Zeisel brain reference (major non-immune types)

Loaded locally from the drive. `Class` gives 7 major types (Neurons,
Oligos, Astrocytes, Vascular, Ependymal, PeripheralGlia, Immune). It's
coarse (neurons aren't sub-typed), but that's fine — its job is to absorb
the non-immune signal so the immune proportions aren't inflated. We cap
cells per class and let `.X` (raw counts) flow into `.layers['counts']`.

In [ ]:
MAX_PER_TYPE = 2000
brain = sc.read_h5ad(LINN_H5AD)
brain.var_names_make_unique()
brain.obs_names_make_unique()
brain.layers['counts'] = brain.X.copy()   # Linnarsson X is raw integer counts
BRAIN_LABEL_COL = 'Class'

brain = dv.subsample_per_type(brain, BRAIN_LABEL_COL, n_per=MAX_PER_TYPE)
print('brain reference:', brain.shape)
print(brain.obs[BRAIN_LABEL_COL].value_counts())

## 3 · Build the combined reference

Strip the brain reference's own `Immune` class, intersect genes with the
atlas, graft the fine immune identities (prefixed `MYE:`), and cap cells per
type. The printed *dropped* labels should be exactly `['Immune']`.

In [ ]:
if COMBINED_H5AD.exists():
    ref = sc.read_h5ad(COMBINED_H5AD)
    print('loaded cached combined reference', ref.shape)
else:
    brain_kept, dropped = dv.strip_myeloid(brain, BRAIN_LABEL_COL)
    print('dropped brain immune labels:', dropped)   # expect ['Immune']

    ref = dv.build_combined_reference(
        brain_kept, atlas,
        brain_label_col=BRAIN_LABEL_COL,
        myeloid_label_col=MYELOID_LABEL_COL,
        myeloid_prefix='MYE:')
    ref = dv.subsample_per_type(ref, 'cell_type', n_per=MAX_PER_TYPE)
    ref.write_h5ad(COMBINED_H5AD)
    print('saved combined reference', ref.shape)

print('\ncell types (', ref.obs.cell_type.nunique(), '):')
print(ref.obs.cell_type.value_counts())

## 4 · cell2location reference regression (per-cell-type signatures)

Select informative genes, then estimate reference expression signatures
(`inf_aver`). `ref_source` is the batch key so Allen-vs-Chhatbar technical
differences are absorbed. **GPU strongly recommended**; the epoch count
below is reduced for CPU.

In [ ]:
import cell2location
from cell2location.models import RegressionModel
from cell2location.utils.filtering import filter_genes

ref.X = ref.layers['counts'].copy()
selected = filter_genes(ref, cell_count_cutoff=5,
                        cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)
ref = ref[:, selected].copy()

RegressionModel.setup_anndata(ref, batch_key='ref_source', labels_key='cell_type')
reg = RegressionModel(ref)
REG_EPOCHS = 250   # cheap even on CPU; bump to ~400 if signatures look noisy
reg.train(max_epochs=REG_EPOCHS, accelerator=ACCELERATOR)

ref = reg.export_posterior(
    ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500,
                        'accelerator': ACCELERATOR})
C2L_REF_DIR.mkdir(parents=True, exist_ok=True)
reg.save(str(C2L_REF_DIR), overwrite=True)

# per-cell-type expression signatures
if 'means_per_cluster_mu_fg' in ref.varm:
    inf_aver = ref.varm['means_per_cluster_mu_fg'].copy()
else:
    inf_aver = ref.var[[c for c in ref.var.columns
                        if 'means_per_cluster_mu_fg' in c]].copy()
inf_aver.columns = [c.replace('means_per_cluster_mu_fg_', '')
                    for c in inf_aver.columns]
inf_aver.to_csv(REF_DIR / 'inf_aver_signatures.csv')
print('signatures:', inf_aver.shape)
inf_aver.iloc[:3, :5]

## 5 · Spatial mapping on the Visium data

Restrict to genes shared with the signatures, then fit cell2location. On
CPU this is the slow step — reduce sections or move to GPU. `batch_key` is
the Visium sample so each section gets its own detection efficiency.

In [ ]:
from cell2location.models import Cell2location

assert H5AD.exists(), f'mount the processing volume: {H5AD}'
vis = sc.read_h5ad(H5AD)
# raw counts into .X (cell2location needs counts)
if COUNT_LAYER in vis.layers:
    vis.X = vis.layers[COUNT_LAYER].copy()
vis.var_names_make_unique()

shared = [g for g in inf_aver.index if g in vis.var_names]
print(f'shared genes vis n signatures: {len(shared)}')
vis = vis[:, shared].copy()
sig = inf_aver.loc[shared]

Cell2location.setup_anndata(vis, batch_key=SAMPLE_KEY)
mod = Cell2location(
    vis, cell_state_df=sig,
    N_cells_per_location=30,   # ~cells per 55um spot; tune to your tissue
    detection_alpha=20,
)
MAP_EPOCHS = 30000 if ACCELERATOR == 'cuda' else 5000   # restore 30000 on GPU
mod.train(max_epochs=MAP_EPOCHS, batch_size=None,
          train_size=1, accelerator=ACCELERATOR)

vis = mod.export_posterior(
    vis, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs,
                        'accelerator': ACCELERATOR})
C2L_VIS_DIR.mkdir(parents=True, exist_ok=True)
mod.save(str(C2L_VIS_DIR), overwrite=True)
print('done; obsm keys:', list(vis.obsm.keys()))

## 6 · Proportions → `.obs`, spatial maps

Normalize the q05 absolute abundances to per-spot fractions, write them to
`.obs` as `prop_<cell_type>`, and map the microglia/myeloid states
spatially (same plotting device as 03/04 and 03/08).

In [ ]:
frac = dv.abundance_to_fractions(vis)
dv.add_fractions_to_obs(vis, frac)
frac.to_csv(TBL / 'per_spot_fractions.csv')

# aggregate myeloid fraction + dominant state per spot
mye_cols = [c for c in frac.columns if c.startswith('MYE:')]
vis.obs['myeloid_frac'] = frac[mye_cols].sum(axis=1).values
vis.obs['dominant_type'] = frac.idxmax(axis=1).values
print('mean composition (top 15):')
print(frac.mean().sort_values(ascending=False).head(15).round(3))
vis.write_h5ad(REF_DIR / 'ST_BRICHOS_deconvolved.h5ad')

In [ ]:
# spatial maps of a few key states (aggregate myeloid + top myeloid states)
to_map = ['myeloid_frac'] + [c for c in frac[mye_cols].mean()
                             .sort_values(ascending=False).head(3).index]
for feat in to_map:
    col = feat if feat == 'myeloid_frac' else f'prop_{feat}'
    vmax = float(np.nanpercentile(vis.obs[col], 99)) or 1e-6
    for treat in ['WT', 'PBS', 'BRICHOS']:
        sub = vis[vis.obs[TREATMENT_KEY] == treat]
        if sub.n_obs == 0:
            continue
        libs = sub.obs[SAMPLE_KEY].unique()
        fig, axes = plt.subplots(1, len(libs), figsize=(3 * len(libs), 3.2), squeeze=False)
        for ax, lib in zip(axes.flat, libs):
            sl = sub[sub.obs[SAMPLE_KEY] == lib]
            sp = vis.uns['spatial'][lib]
            sf = sp['scalefactors']['tissue_hires_scalef']
            ax.imshow(sp['images']['hires'], origin='upper')
            xy = sl.obsm['spatial'] * sf
            ax.scatter(xy[:, 0], xy[:, 1], c=sl.obs[col].values, s=0.6,
                       cmap='magma', vmin=0, vmax=vmax, edgecolors='none')
            ax.set_title(f'{lib} [{treat}]', fontsize=8, loc='left')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
        fig.suptitle(f'{feat}: proportion', fontsize=10)
        fig.tight_layout()
        fig.savefig(FIG / f'deconv_{feat.replace(":", "_")}_{treat}.png', dpi=200, bbox_inches='tight')
        plt.show()

## 7 · Region × treatment composition & BRI-vs-PBS shifts

Mean per-spot fraction by region×treatment, and a per-state test of whether
BRICHOS shifts composition vs PBS (sample-level Mann–Whitney, mirroring
03/08's responder framing). Focus on the `MYE:` states.

In [ ]:
comp = dv.region_composition(vis, frac, REGION_KEY, TREATMENT_KEY)
comp.to_csv(TBL / 'region_treatment_composition.csv')

# heatmap: region x myeloid-state mean fraction (PBS only, as the baseline)
pbs = comp.xs('PBS', level=TREATMENT_KEY)[mye_cols]
fig, ax = plt.subplots(figsize=(0.5 * len(mye_cols) + 3, 0.4 * pbs.shape[0] + 2))
im = ax.imshow(pbs.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(mye_cols))); ax.set_xticklabels(mye_cols, rotation=90, fontsize=6)
ax.set_yticks(range(pbs.shape[0])); ax.set_yticklabels(pbs.index, fontsize=8)
ax.set_title('Myeloid-state mean fraction by region (PBS)')
fig.colorbar(im, ax=ax, shrink=0.6)
fig.tight_layout(); fig.savefig(FIG / 'deconv_myeloid_region_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

# sample-level fraction per region x state, BRI vs PBS
df = frac.copy()
df[[SAMPLE_KEY, REGION_KEY, TREATMENT_KEY]] = vis.obs[[SAMPLE_KEY, REGION_KEY, TREATMENT_KEY]].values
samp = df.groupby([REGION_KEY, TREATMENT_KEY, SAMPLE_KEY], observed=True)[mye_cols].mean().reset_index()

rows = []
for region in samp[REGION_KEY].unique():
    r = samp[samp[REGION_KEY] == region]
    bri = r[r[TREATMENT_KEY] == 'BRICHOS']; pbs_ = r[r[TREATMENT_KEY] == 'PBS']
    if len(bri) < 2 or len(pbs_) < 2:
        continue
    for st in mye_cols:
        try:
            _, p = mannwhitneyu(bri[st], pbs_[st])
        except ValueError:
            p = np.nan
        rows.append(dict(region=region, state=st,
                         mean_pbs=pbs_[st].mean(), mean_bri=bri[st].mean(),
                         delta=bri[st].mean() - pbs_[st].mean(), mwu_p=p))
shift = pd.DataFrame(rows).sort_values('mwu_p')
shift.to_csv(TBL / 'myeloid_state_BRIvPBS_shift.tsv', sep='\t', index=False)
shift.head(20).round(4)

## 8 · Interpretation (CPU run, 2026-09-16)

Combined reference: **6 brain classes (Linnarsson) + 12 immune states (Chhatbar `MYE:`)**, 25,113 cells, 15,428 genes shared with the Visium. Full run on the Tailscale peer (cell2location 0.1.5, CPU): reference regression 250 epochs, spatial mapping 5,000 epochs. Outputs in `results/tables/deconvolution/` and `results/figures/manuscript/deconv_*`.

**Trustworthy**
- Structural background is anatomically correct — Neurons 38 %, Astro 17 %, Oligo 8 %, Ependymal 5.5 %, Vascular 4.3 %, PeripheralGlia 3.4 %. Region composition tracks anatomy (Corpus callosum oligo-rich 24 %; Meninges astro+vascular-rich; ventricular area ependymal-rich).
- `MYE:Microglia` is the dominant, spatially-structured immune signal (peaks in Meninges / Olfactory areas); `CAM`/`MDM`/`Granulocyte` peak at the ventricular–CSF interface. Biologically coherent.

**Caveats — do not over-read the immune numbers**
- Total immune ≈ **24 % per spot is inflated**: cell2location spread mass across the 12 similar rare immune signatures. The adaptive/peripheral states (CD4 Th17, CD8 TRM, MAIT, NK, Treg, TEx, cDC1, Plasmablasts) sit at a flat ~1.5–2 % floor with little spatial structure → **signature bleed, not real resident cells**. Interpret **Microglia + CAM** (and MDM/granulocyte at interfaces) only.
- BRI-vs-PBS Mann-Whitney p-values hit the discrete floor (0.0238) → few biological replicates. Exploratory; needs per-sample aggregation + FDR.

**Biological signal (cautious, consistent)**
- *Injection effect:* both BRICHOS and PBS show more microglia / fewer neurons than untreated WT across regions.
- *Treatment effect:* **BRICHOS consistently has slightly lower microglia/myeloid fraction than PBS** across most regions (Caudoputamen 0.026 vs 0.028; Meninges 0.034 vs 0.037; Olfactory 0.033 vs 0.039) — BRICHOS partially normalizes the injection-induced microglial expansion back toward WT. Small but uniform-direction; dovetails with the BA4_MIC-7 resilience result in `03/08`.

**Next steps**
- Prune the reference to plausible CNS-resident myeloid types (Microglia, CAM, MDM, ±granulocyte/cDC1) or collapse the T/NK/B states, then re-run — should sharpen the microglia estimate and cut the 24 % over-assignment.
- Per-panel percentile scaling on the spatial maps (microglia fraction varies only ~0.02–0.04, washed out over H&E).
- Proper stats: aggregate to per-sample means → BRI vs PBS with FDR (or a mixed model).
- Correlate per-spot `prop_MYE:Microglia` / `prop_MYE:CAM` with the `03/08` MIC-7 resilience score.
